# LSTM & Bidirectional RNN — Beginner Notebook

This is the follow-up to the SimpleRNN sentiment analysis notebook. Here we
upgrade the same movie-review classifier in two steps:

1. **LSTM** — a smarter RNN cell with a protected long-term memory (the
   "cell state") and three gates that control what gets kept, added, or
   shown.
2. **Bidirectional** — wraps a layer so it reads the sentence **both**
   forward and backward, then combines what each direction learned.

We reuse the exact same IMDB movie review data and pipeline as before, so
we can fairly compare **SimpleRNN vs LSTM vs Bidirectional LSTM** side by
side.

Run each cell top to bottom. Every code cell has a markdown explanation
above it in simple words.

## Step 1 — Import the tools we need

Same as before, plus two new layers:
- `LSTM` — the upgraded memory cell.
- `Bidirectional` — a wrapper that runs any recurrent layer in both
  directions.

In [2]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Bidirectional, Dense

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


## Step 2 — Load and prepare the data (same as Part 1)

We keep the same settings as the SimpleRNN notebook:
- Only the **10,000 most common words**.
- Every review padded/cut to **200 words**.

This keeps the comparison between models fair — the only thing changing
is the recurrent layer itself.

In [3]:
VOCAB_SIZE = 10000
MAX_LEN = 200

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

x_train = pad_sequences(x_train, maxlen=MAX_LEN)
x_test = pad_sequences(x_test, maxlen=MAX_LEN)

print("Training reviews:", len(x_train))
print("Test reviews:", len(x_test))

Training reviews: 25000
Test reviews: 25000


## Step 3 — Build the LSTM model

The only change from a `SimpleRNN` model is swapping one layer:

- **`Embedding`** — same as before, turns word IDs into 32-number meaning
  vectors.
- **`LSTM(32)`** — instead of `SimpleRNN(32)`. It still outputs a 32-number
  hidden state, but internally it now keeps a separate **cell state** that
  travels across words mostly unchanged, protected by three gates:
  - **Forget gate** — decides what old information to drop.
  - **Input gate** — decides what new information to add.
  - **Output gate** — decides what to reveal as the hidden state.
- **`Dense(1, activation='sigmoid')`** — same as before, turns the final
  hidden state into a 0–1 sentiment score.

Everything else (Embedding size, Dense layer, compile settings) is
identical to the SimpleRNN model.

In [4]:
lstm_model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=32, input_length=MAX_LEN),
    LSTM(32),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

lstm_model.summary()

c:\Users\Primax\anaconda3\envs\venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Step 4 — Train and evaluate the LSTM model

Same training recipe as before: 5 epochs, batches of 128 reviews, 20% held
out for validation.

In [5]:
lstm_model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

lstm_loss, lstm_accuracy = lstm_model.evaluate(x_test, y_test)
print(f"\nLSTM Test Accuracy: {lstm_accuracy * 100:.2f}%")

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 15s 80ms/step - accuracy: 0.7329 - loss: 0.5104 - val_accuracy: 0.8478 - val_loss: 0.3609
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 12s 77ms/step - accuracy: 0.8902 - loss: 0.2755 - val_accuracy: 0.8734 - val_loss: 0.3061
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 12s 79ms/step - accuracy: 0.9240 - loss: 0.2022 - val_accuracy: 0.8690 - val_loss: 0.3283
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 20s 74ms/step - accuracy: 0.9442 - loss: 0.1575 - val_accuracy: 0.8700 - val_loss: 0.3327
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 11s 73ms/step - accuracy: 0.9555 - loss: 0.1304 - val_accuracy: 0.8538 - val_loss: 0.3948
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.8401 - loss: 0.4302

LSTM Test Accuracy: 84.01%


## Step 5 — Build the Bidirectional LSTM model

Now we upgrade again. `Bidirectional(LSTM(32))` runs **two** LSTMs:

- One reads the review **left to right** (forward), just like before.
- A second, separate one reads the same review **right to left**
  (backward).
- Their two 32-number outputs are **concatenated** into a single 64-number
  vector before being passed on.

This means the model sees each word with context from *both* directions —
useful for sentences where a later word (like "excellent") changes how an
earlier phrase (like "a slow start") should be read.

Notice the code change is just **one line** — wrapping `LSTM(32)` in
`Bidirectional(...)`. Everything else stays the same.

In [6]:
bilstm_model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=32, input_length=MAX_LEN),
    Bidirectional(LSTM(32)),
    Dense(1, activation='sigmoid')
])

bilstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

bilstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Bidirectional RNN

In [ ]:
birnn_model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=32, input_length=MAX_LEN),
    Bidirectional(SimpleRNN(32)),
    Dense(1, activation='sigmoid')
])

bilstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

bilstm_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [16]:
birnn_model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

birnn_loss, birnn_accuracy = birnn_model.evaluate(x_test, y_test)
print(f"\nBidirectional RNN Test Accuracy: {birnn_accuracy * 100:.2f}%")

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 50ms/step - accuracy: 0.9806 - loss: 0.0666 - val_accuracy: 0.8452 - val_loss: 0.4366
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 49ms/step - accuracy: 0.9881 - loss: 0.0453 - val_accuracy: 0.8564 - val_loss: 0.5045
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 11s 53ms/step - accuracy: 0.9923 - loss: 0.0300 - val_accuracy: 0.8476 - val_loss: 0.5378
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 50ms/step - accuracy: 0.9718 - loss: 0.0773 - val_accuracy: 0.8410 - val_loss: 0.5318
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 45ms/step - accuracy: 0.9919 - loss: 0.0296 - val_accuracy: 0.8416 - val_loss: 0.5702
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.8309 - loss: 0.6105

Bidirectional RNN Test Accuracy: 83.09%


## Step 6 — Train and evaluate the Bidirectional LSTM model

Same recipe again, so the comparison with the plain LSTM stays fair.

In [8]:
bilstm_model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

bilstm_loss, bilstm_accuracy = bilstm_model.evaluate(x_test, y_test)
print(f"\nBidirectional LSTM Test Accuracy: {bilstm_accuracy * 100:.2f}%")

Epoch 1/5


157/157 ━━━━━━━━━━━━━━━━━━━━ 24s 127ms/step - accuracy: 0.7391 - loss: 0.5201 - val_accuracy: 0.8206 - val_loss: 0.4066
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 18s 117ms/step - accuracy: 0.8878 - loss: 0.2817 - val_accuracy: 0.8694 - val_loss: 0.3084
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 19s 120ms/step - accuracy: 0.9270 - loss: 0.2001 - val_accuracy: 0.8600 - val_loss: 0.3186
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 19s 119ms/step - accuracy: 0.9448 - loss: 0.1612 - val_accuracy: 0.8682 - val_loss: 0.3375
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 19s 123ms/step - accuracy: 0.9564 - loss: 0.1279 - val_accuracy: 0.8724 - val_loss: 0.3710
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.8587 - loss: 0.4182

Bidirectional LSTM Test Accuracy: 85.87%


## Step 7 — Compare three two models

A quick side-by-side of test accuracy. Results can vary a little each run
(random initialization), but this gives a sense of whether the extra gates
and the extra reading direction actually helped on this dataset.

In [17]:
print("Model Comparison")
print("-" * 40)
print(f"LSTM:                {lstm_accuracy * 100:.2f}%")
print(f"Bidirectional LSTM:  {bilstm_accuracy * 100:.2f}%")
print(f"Bidirectional RNN:   {birnn_accuracy * 100:.2f}%")


Model Comparison
----------------------------------------
LSTM:                84.01%
Bidirectional LSTM:  85.87%
Bidirectional RNN:   83.09%


## Step 8 — Try both models on our own sentences

Same encoding helper as the SimpleRNN notebook — it converts a plain
sentence into the word-ID format the models were trained on.

In [10]:
word_index = imdb.get_word_index()

def encode_review(text):
    words = text.lower().split()
    encoded = [1]  # 1 = "start of review" marker
    for word in words:
        idx = word_index.get(word, 2) + 3
        encoded.append(idx if idx < VOCAB_SIZE else 2)
    return encoded

def predict_sentiment(text, model, model_name):
    encoded = pad_sequences([encode_review(text)], maxlen=MAX_LEN)
    score = model.predict(encoded, verbose=0)[0][0]
    label = "Positive 🙂" if score > 0.5 else "Negative 🙁"
    print(f"[{model_name}] \"{text}\" -> {label} (score: {score:.2f})")

## Step 9 — Test with a POSITIVE example

We run the same sentence through both models so we can compare their
predictions directly.

In [20]:
sentence = "the movie was good, and I really enjoyed the performances of the actors."
predict_sentiment(sentence, lstm_model, "LSTM")
predict_sentiment(sentence, bilstm_model, "Bidirectional LSTM")
predict_sentiment(sentence, birnn_model, "Bidirectional RNN")

[LSTM] "the movie was good, and I really enjoyed the performances of the actors." -> Positive 🙂 (score: 0.92)
[Bidirectional LSTM] "the movie was good, and I really enjoyed the performances of the actors." -> Positive 🙂 (score: 0.94)
[Bidirectional RNN] "the movie was good, and I really enjoyed the performances of the actors." -> Positive 🙂 (score: 1.00)


## Step 10 — Test with a NEGATIVE example

In [22]:
sentence = "this movie was like a funny joke, but it failed to deliver any real laughs."
predict_sentiment(sentence, lstm_model, "LSTM")
predict_sentiment(sentence, bilstm_model, "Bidirectional LSTM")
predict_sentiment(sentence, birnn_model, "Bidirectional RNN")


[LSTM] "this movie was like a funny joke, but it failed to deliver any real laughs." -> Negative 🙁 (score: 0.26)
[Bidirectional LSTM] "this movie was like a funny joke, but it failed to deliver any real laughs." -> Negative 🙁 (score: 0.19)
[Bidirectional RNN] "this movie was like a funny joke, but it failed to deliver any real laughs." -> Negative 🙁 (score: 0.25)


## Step 11 — Test with a trickier, mixed-signal example

This is the kind of sentence Bidirectional models are meant to handle
better — a positive word arrives late, after an early negative-sounding
phrase.

In [23]:
sentence = "this movie was boring but had some good moments."
predict_sentiment(sentence, lstm_model, "LSTM")
predict_sentiment(sentence, bilstm_model, "Bidirectional LSTM")
predict_sentiment(sentence, birnn_model, "Bidirectional RNN")

[LSTM] "this movie was boring but had some good moments." -> Negative 🙁 (score: 0.22)
[Bidirectional LSTM] "this movie was boring but had some good moments." -> Negative 🙁 (score: 0.20)
[Bidirectional RNN] "this movie was boring but had some good moments." -> Negative 🙁 (score: 0.04)


### A note on these predictions

As with the SimpleRNN notebook, don't be surprised if a short custom
sentence gets misclassified by one or both models. These models were
trained on full-length movie reviews (up to 200 words), so a 6–8 word
sentence is quite different from what they mostly learned on. This isn't a
sign the code is broken — it's a real, expected limit of a small model
trained for only 5 epochs on a fixed vocabulary.

## Step 12 — Try your own sentence

Change the text below and re-run to compare both models on your own
example.

In [25]:
sentence = "the actor was good but the story had a lot of plot holes and inconsistencies."
predict_sentiment(sentence, lstm_model, "LSTM")
predict_sentiment(sentence, bilstm_model, "Bidirectional LSTM")
predict_sentiment(sentence, birnn_model, "Bidirectional RNN")

[LSTM] "the actor was good but the story had a lot of plot holes and inconsistencies." -> Negative 🙁 (score: 0.36)
[Bidirectional LSTM] "the actor was good but the story had a lot of plot holes and inconsistencies." -> Negative 🙁 (score: 0.39)
[Bidirectional RNN] "the actor was good but the story had a lot of plot holes and inconsistencies." -> Negative 🙁 (score: 0.48)


## Recap — what we just built

1. **Reused the data** — same IMDB reviews, padded to 200 words.
2. **Built an LSTM model** — `Embedding` → `LSTM` → `Dense`. The LSTM
   cell keeps a protected long-term "cell state," guarded by forget/input/
   output gates, so it forgets less over long text than a `SimpleRNN`.
3. **Built a Bidirectional LSTM model** — `Embedding` →
   `Bidirectional(LSTM)` → `Dense`. This reads the review both forward
   and backward and concatenates the two views before deciding.
4. **Trained and compared** both models under identical settings.
5. **Tested on real sentences** — including one designed to need context
   from both directions.

**Key takeaway:** each upgrade — `SimpleRNN` → `LSTM` →
`Bidirectional(LSTM)` — was only a one-line change in the model
definition, but each one gives the network a meaningfully different way
of handling context and memory.